In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass

def is_gpu_available():
    # Проверяем код возврата системной команды nvidia-smi
    # Если 0 — карта есть и драйверы работают
    return os.system('nvidia-smi > /dev/null 2>&1') == 0

@dataclass
class Config:
    DATA_DIR: Path = Path("data")
    target: str = "target"
    
    # Определяем наличие GPU один раз
    has_gpu = is_gpu_available()
    
    device = "cuda" if has_gpu else "cpu"
    task_type = "GPU" if has_gpu else "CPU"

    # catboost settings 
    random_seed: int = 42
    cat_features: tuple[str] = ("parentname1", "subjectname2") # ЗДЕСЬ ЛИБО ЗАДАЕМ ЧЕРЕЗ ВХОДНОЙ ФАЙЛ, ЛИБО УБИРАЕМ ЭТО
    verbose: int = 100
    eval_metric: str = "PRAUC" # МЕТРИКА

cfg = Config()
print(f"Current device: {cfg.device} (Task type: {cfg.task_type})")

In [ ]:

train_pool = Pool(X_train, y_train, cat_features=cfg.cat_features) # МОЖНО  НЕ ЧЕРЕЗ POOL
val_pool = Pool(X_val, y_val, cat_features=cfg.cat_features) # МОЖНО  НЕ ЧЕРЕЗ POOL

model = CatBoostClassifier(
    verbose=cfg.verbose,
    eval_metric=cfg.eval_metric,
    task_type=cfg.task_type,
    random_state=cfg.random_seed
    )

model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)